# 🎯 Avatar Analysis với Ollama llava:13b
**Phân tích chân dung khách hàng qua ảnh avatar — song song 4 workers**

Yêu cầu: `ollama pull llava:13b` đã chạy xong

In [13]:
import base64
import requests
from PIL import Image
from io import BytesIO
import time

# =========================
# CONFIG
# =========================
OLLAMA_URL = "http://localhost:11434/api/generate"
#MODEL_NAME = "qwen2.5vl:3b"
MODEL_NAME = "minicpm-v4.6"

IMAGE_URL = (
    "https://studiochupanhdep.com/Upload/Images/Album/"
    "chan-dung-beauty-05.jpg"
)

# =========================
# LOAD + OPTIMIZE IMAGE
# =========================
print("Đang tải ảnh từ internet...")

response = requests.get(IMAGE_URL, timeout=30)
response.raise_for_status()

img = Image.open(BytesIO(response.content))

# resize cho CPU inference
img.thumbnail((448, 448))

if img.mode != "RGB":
    img = img.convert("RGB")

buf = BytesIO()

img.save(
    buf,
    format="JPEG",
    quality=80,
    optimize=True,
)

b64 = base64.b64encode(buf.getvalue()).decode()

print(
    f"Ảnh đã tối ưu: {img.size}, "
    f"dung lượng base64: {len(b64)//1024}KB"
)

# =========================
# PROMPT
# =========================
MARKETING_PROMPT = """
Extract customer signals from this image.

Rules:

* Output only observable evidence and weak inference.
* No marketing advice.
* No personality inference.
* Use broad ranges.
* Mark uncertainty.

Return:

1. PEOPLE

* Count
* Group type
* Age bands
* Gender presentation
* Interaction

2. APPEARANCE

* Clothing
* Formality
* Colors
* Grooming
* Accessories
* Visible brands

3. CONTEXT

* Location type
* Indoor/outdoor
* Time/weather clues
* Crowd level
* Lighting

4. BEHAVIOR

* Activity
* Objects used
* Food/drinks
* Device usage

5. CUSTOMER SIGNALS

* Leisure ↔ Work
* Social ↔ Individual
* Visit pace
* Spending cues (only if visible)

Format:
Value | Evidence | Confidence (0–100)



"""

# =========================
# REQUEST
# =========================
payload = {
    "model": MODEL_NAME,
    "prompt": MARKETING_PROMPT,
    "images": [b64],
    "stream": False,
    "keep_alive": -1,
    "options": {
        "temperature": 0.0,
        "top_p": 0.1,
        "num_predict": 350,
    },
}

print(
    f"\n[Marketing Stage] "
    f"{MODEL_NAME} đang phân tích..."
)

try:
    t0 = time.time()

    resp = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=120,
    )

    resp.raise_for_status()

    elapsed = time.time() - t0

    result = (
        resp.json()
        .get("response", "")
        .strip()
    )

    print(f"\n⏱️ Thời gian xử lý: {elapsed:.2f}s")
    print("\n📊 Dữ liệu Marketing thu được:\n")
    print(result)

except requests.Timeout:
    print("\n❌ Timeout - tăng timeout hoặc giảm ảnh")

except Exception as e:
    print(f"\n❌ Lỗi: {e}")



Đang tải ảnh từ internet...
Ảnh đã tối ưu: (373, 448), dung lượng base64: 20KB

[Marketing Stage] minicpm-v4.6 đang phân tích...

⏱️ Thời gian xử lý: 8.41s

📊 Dữ liệu Marketing thu được:

## PEOPLE

### People

* Count: 1
* Group type: female
* Age bands: young adult
* Gender presentation: feminine
* Interaction: posing, hand near face

## APPEARANCE

### Appearance

* Clothing: turtleneck top
* Formality: semi-formal
* Colors: black, red (text)
* Grooming: neatly styled hair
* Accessories: none
* Visible brands: none

## CONTEXT

### Context

* Location type: studio
* Indoor/outdoor: indoor
* Time/weather clues: not applicable
* Crowd level: none
* Lighting: controlled lighting

## BEHAVIOR

### Behavior

* Activity: posing, hand near face
* Objects used: none
* Food/drinks: not present
* Device usage: not present

## CUSTOMER SIGNALS

### Leisure ↔ Work

Not applicable

### Social ↔ Individual

Social interaction for individual
